# ARGUS demo snapshot (current pipeline)
Run all three cells with a GPU. Cell 1 is the unchanged setup. Cell 2 is the unchanged pipeline, trimmed to train one encoder and stop. Cell 3 writes `argus_demo_snapshot_v2.json` to the output, which `site/index.html` replays via *Load snapshot*.

In [ ]:
# ===== CELL 1 v2 / SETUP -- multi-species, shot-strategy aware =====
# Replaces arguswala_setup.py. Run BEFORE arguswala_v2.py.
#
# WHAT CHANGED vs v1, and why:
#
#  (1) MULTI-TARGET. v1 hard-wired one TARGET. Everything per-species is now behind
#      build_target(sp), so the same pipeline evaluates N species. This pays the debt
#      arguswala.py:375 admits to ("one target species is not a validated finding").
#
#  (2) SPECIES CACHE COMPUTED ONCE. v1 embedded 30 candidate species per target. With
#      10 targets that is 10x redundant Perch inference -- the dominant cost. The cache
#      is now global; build_target() only ranks against it. ~10x saving on setup.
#
#  (3) SHOT-SELECTION STRATEGY is now a variable, not an assumption. v1 always took the
#      5 highest-`rating` recordings. DCASE always takes the first 5. Nolasco et al.
#      (2023) sec.4.1 report that unrepresentative support sets were the dominant failure
#      cause their expert annotators identified, and call better selection an open
#      problem. So it becomes an axis we measure instead of a default we inherit.
#
#  (4) PER-SPECIES COVARIATES are recorded at build time (decoy proximity, stereotypy,
#      spectral centre/bandwidth, call duration, n recordings). Nolasco et al. tried
#      multivariate regression to find what predicts task difficulty and FAILED to find
#      any consistent factor. These covariates are free at runtime and impossible to
#      reconstruct afterwards -- logging them is what makes that analysis possible later.
#
#  (5) The held-out pool is now derived from the shots actually chosen, not from a
#      positional slice. Under a non-'first' strategy, files[N_SHOT:] would have leaked
#      support recordings into the evaluation pool.
import os, glob, time
import numpy as np
import pandas as pd
import librosa
import tensorflow_hub as hub
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans

# ---------------------------------------------------------------- ablation ladder
# Each rung is independently switchable so marginal contributions are measurable.
# Rung 0 is the original pipeline; nothing below changes it when all flags are False.
VERIFIER = "logreg"        # 'logreg' (rung 0) | 'proto' (rung 1, stage 2)

# ------------------------------------------------------------------ configuration
SR = 22050
SEG_SEC = 1.0
CLEN = int(SEG_SEC * SR)

N_SHOT          = 5
SHOT_STRATEGY   = "rated"     # 'rated' | 'first' | 'random' | 'diverse'
N_DECOYS        = 4
N_CANDIDATES    = 30          # non-target species considered for decoy ranking
CALIB_PER_SPECIES = 6
DECOY_RECORDINGS  = 20
N_CALIB_BEDS, N_EVAL_BEDS = 4, 10
SNR_LEVELS = (9, 6, 3, 0, -3, -6)

# Targets are PRE-REGISTERED: chosen by a rule, before any result is seen.
# Rule: species with >= MIN_RECS recordings, ranked by recording count, top N_TARGETS,
# with the original study species forced in first. Deterministic -- no cherry-picking.
PRIMARY_TARGET = "whbsho3"    # White-bellied Sholakili, the original study species
N_TARGETS      = 10
MIN_RECS       = N_SHOT + 4   # 5 shots + >=4 genuinely held out

SMOKE_TEST = False            # 15 Aug 2026: timing measured (10 min/species/3seeds),
                              # config locked (PCEN/logreg/no-tfa/no-hn all confirmed
                              # winners in the ladder), noise floor known (0.025-0.035).
                              # Centrepiece run: full N_TARGETS species, no more reason
                              # to stay on the 2-species smoke test.

# Added 14 Aug 2026, after the first real run: whbsho3 stage-1-alone AUC came back 0.517
# in a run that excludes 2 species from the encoder bank (this multi-target pipeline),
# vs. the original single-species baseline of 0.459, which only ever excluded whbsho3.
# ENCODER_SEEDS (widened separately, in CELL 2) tests training-run-to-training-run noise
# on a FIXED bank. This flag tests the OTHER candidate cause: does the bank's COMPOSITION
# itself change when only 1 species is excluded instead of 2? Set True to force
# TARGET_LIST down to [PRIMARY_TARGET] alone -- the exact exclusion set the original
# baseline used -- so build_banks() in CELL 2 produces a directly comparable bank.
# Re-run CELL 1 then CELL 2 after flipping this; no kernel restart needed, everything
# downstream is rebuilt from these globals. CELL 3/4 are not needed for this comparison.
SINGLE_TARGET_COMPARISON = False

# Added 29 Aug 2026 for CELL 6 (Perch distillation). Selects which embedding space the
# STAGE-2 VERIFIER is trained and run in:
#   "perch"     -- Perch v2's 1536-d embeddings. Every result published before this flag
#                  existed used this, and it stays the default.
#   "distilled" -- a ~445k-param CNN trained (CELL 6) to reproduce Perch's embeddings.
#                  At detection time NO Perch inference runs at all.
#
# DELIBERATELY NARROW. Only the verifier's embeddings switch. Decoy ranking
# (SPECIES_CACHE, perch_embed) and 'diverse' shot selection stay on Perch under BOTH
# settings, because those choose WHAT THE TEST IS -- which species count as decoys, and
# which five calls are the support set. If the student picked its own decoys, the two arms
# would be scored against different test sets and the comparison would be meaningless.
# This is the same trap the literature review flagged for the Perch v1->v2 upgrade
# ("change Perch, and the decoy set changes, and the difficulty of the specificity test
# changes with it"). Holding the test fixed is what makes teacher-vs-student a clean A/B.
EMBED_BACKEND = "perch"
DISTILL_EMBED_CTX = None      # set by CELL 6 to the trained student's embedder

def verifier_embed_ctx(y22, centres_sec):
    """Embeddings the stage-2 verifier is FIT on and SCORED with. y22 is a 22050 Hz
    recording; centres_sec are event centres in seconds. Dispatches on EMBED_BACKEND so
    that build_target()'s verifier fit and CELL 2's detect_pv() always agree about which
    embedding space they are in -- a mismatch there would fit the classifier in one space
    and score candidates in another, which produces plausible-looking nonsense rather
    than an error."""
    if EMBED_BACKEND == "perch":
        return perch_embed_ctx(to32k(y22), centres_sec)
    if EMBED_BACKEND == "distilled":
        assert DISTILL_EMBED_CTX is not None, \
            "EMBED_BACKEND='distilled' but CELL 6 has not set DISTILL_EMBED_CTX"
        return DISTILL_EMBED_CTX(y22, centres_sec)
    raise ValueError(f"unknown EMBED_BACKEND {EMBED_BACKEND!r}")

root = os.path.dirname(glob.glob('/kaggle/input/**/train_metadata.csv', recursive=True)[0])
AUD = os.path.join(root, 'train_audio')
SS  = os.path.join(root, 'unlabeled_soundscapes')
meta = pd.read_csv(os.path.join(root, 'train_metadata.csv'))

# Recordist identity column, for the recordist-disjoint confound check (CELL 5, roadmap
# "owed" list: rules out "did it learn the microphone, not the bird"). BirdCLEF inherits
# Xeno-Canto's schema, where the recordist's name is the 'author' field -- but that is a
# CONVENTION, not something this script has verified against the actual attached dataset,
# so it is printed here rather than assumed silently. If build_target(recordist_disjoint=
# True) raises a KeyError, check this list against the real column name and fix RECORDIST_COL.
RECORDIST_COL = "author"
print(f"train_metadata.csv columns: {meta.columns.tolist()}")
print(f"  RECORDIST_COL = {RECORDIST_COL!r} "
      f"({'present' if RECORDIST_COL in meta.columns else 'NOT FOUND -- fix before using recordist_disjoint'})")

# ------------------------------------------------------------------ audio helpers
def rms(x):
    return float(np.sqrt(np.mean(np.square(x, dtype=np.float64)) + 1e-12))

def ld(p, dur=None):
    y, _ = librosa.load(p, sr=SR, mono=True, duration=dur)
    return y.astype('float32')

def l2n(x):
    x = np.asarray(x, dtype="float32")
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-9)

def cos(a, b):
    return float(a @ b / ((np.linalg.norm(a) + 1e-9) * (np.linalg.norm(b) + 1e-9)))

def loudest_offset(y, n=CLEN):
    if len(y) <= n: return 0
    e = np.square(y, dtype=np.float64); cs = np.concatenate([[0.0], np.cumsum(e)])
    return int(np.argmax(cs[n:] - cs[:-n]))

def peak_windows(y, k=1, n=CLEN):
    """Top-k non-overlapping highest-energy windows of length n."""
    if len(y) <= n: return [np.pad(y, (0, n - len(y))).astype('float32')]
    e = np.square(y, dtype=np.float64); cs = np.concatenate([[0.0], np.cumsum(e)])
    w = cs[n:] - cs[:-n]
    out, taken = [], []
    for _ in range(k):
        pick = next((int(i) for i in np.argsort(w)[::-1] if all(abs(int(i) - t) >= n for t in taken)), None)
        if pick is None: break
        taken.append(pick); out.append(y[pick:pick + n].astype('float32'))
    return out

def build_support_views(recordings, shifts=(-0.25, 0.0, 0.25)):
    out = []
    for y in recordings:
        base = loudest_offset(y)
        for shift in shifts:
            a = int(np.clip(base + shift * SR, 0, max(0, len(y) - CLEN)))
            w = y[a:a + CLEN]
            if len(w) < CLEN: w = np.pad(w, (0, CLEN - len(w)))
            out.append(w.astype('float32'))
    return out

def place(y, pool, rng, spans, snr_db, gap_s=0.4):
    """Mix one random call from `pool` into `y` at a free position -> centre in seconds."""
    for _ in range(80):
        pos = int(rng.integers(0, len(y) - CLEN))
        if any(pos < e + int(gap_s*SR) and pos + CLEN > s - int(gap_s*SR) for s, e in spans):
            continue
        c = pool[int(rng.integers(len(pool)))][:CLEN]
        if len(c) < CLEN: c = np.pad(c, (0, CLEN - len(c)))
        g = rms(y[pos:pos+CLEN]) * (10 ** (float(snr_db)/20)) / (rms(c) + 1e-9)
        y[pos:pos+CLEN] += c.astype('float32') * g
        spans.append((pos, pos + CLEN))
        return (pos + CLEN/2) / SR
    return None

# ------------------------------------------------------------------ Perch
# PERCH 2.0 UPGRADE (Aug 2025 release, arXiv 2508.04665). Three things changed and each
# one silently breaks the v1 code path, so all three are handled explicitly. Verified
# against the official loader: google-research/perch-hoplite, zoo/taxonomy_model_tf.py.
#
#  (a) NO `infer_tf`. Perch 1.0 was a TF-Jax export exposing model.infer_tf(). Perch 2.0
#      exposes only the standard serving signature. perch-hoplite branches on exactly
#      this: `elif hasattr(self.model, 'infer_tf') ... else: # Perch v2 style
#      outputs = self.model.signatures['serving_default'](inputs=rebatched_audio)`.
#      Calling infer_tf on v2 raises AttributeError.
#
#  (b) EMBEDDING WIDTH 1280 -> 1536 (EfficientNet-B1 -> B3). Probed, never assumed.
#      v2 also emits a `spatial_embedding` (5,3,1536) head that must be ignored --
#      `embedding` is the pooled mean vector we want.
#
#  (c) AUDIO IS PEAK-NORMALISED BEFORE INFERENCE. The official wrapper removes DC offset
#      and scales each window to a peak of 0.25 (zoo_interface.normalize_audio,
#      target_peak=0.25). ARGUS v1 fed raw waveforms to Perch and did NEITHER -- so
#      Perch was being given amplitudes outside the distribution it is served with.
#      This affects the EXISTING v1 numbers too, not just the upgrade.
#      Note it does NOT disturb the SNR sweep: peak norm is a per-window scalar multiply,
#      and SNR is a ratio within the window, so planted-call SNR is preserved exactly.
PERCH_VERSION   = "v2"     # "v2" | "v1"  -- recorded in the results CSV
PERCH_PEAK_NORM = 0.25     # None to disable (v1 behaviour); 0.25 matches the official wrapper
PERCH_BACKEND   = "tf"     # "tf" | "onnx" -- see below. Default "tf" -> ZERO behaviour
                           # change to the tested GPU pipeline unless explicitly opted in.

PERCH_SR, PERCH_WIN = 32000, 5.0     # unchanged between versions
PERCH_LEN = int(PERCH_WIN * PERCH_SR)
PERCH_BATCH = 16

_V2_SLUGS = ("perch_v2", "perch_v2_cpu")
def _find_local_perch(prefer_v2):
    """Kaggle attaches models as input dirs. Pick one matching the requested version."""
    hits = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/saved_model.pb', recursive=True)
            if 'bird-vocalization' in p.lower() or 'perch' in p.lower()]
    if not hits: return None
    is_v2 = lambda p: any(s in p.lower() for s in _V2_SLUGS)
    pool = [p for p in hits if is_v2(p) == prefer_v2] or hits
    def _ver(p):
        try: return int(os.path.basename(p))
        except ValueError: return -1
    return max(pool, key=_ver)

def _remote_handle():
    if PERCH_VERSION != "v2":
        # perch_8 in perch-hoplite's config; v8 is the last 1280-d release.
        return ("https://www.kaggle.com/models/google/bird-vocalization-classifier"
                "/tensorFlow2/bird-vocalization-classifier/8")
    # perch-hoplite ships separate GPU and CPU builds at different version numbers.
    try:
        import tensorflow as tf
        gpu = bool(tf.config.list_physical_devices('GPU'))
    except Exception:
        gpu = False
    slug, ver = ("perch_v2", 2) if gpu else ("perch_v2_cpu", 1)
    return ("https://www.kaggle.com/models/google/bird-vocalization-classifier"
            f"/tensorFlow2/{slug}/{ver}")

# ---------------------------------------------------------------- ONNX backend (opt-in)
# For CPU-constrained / edge-hardware experiments (roadmap: ARGUS_Hardware_Deployment_
# Roadmap.md), not for the main GPU pipeline. Uses a pre-converted, community-published,
# Apache-2.0 ONNX export (justinchuby/Perch-onnx on Hugging Face) rather than converting
# ourselves -- same I/O signature, independently verified against the real 413MB weights
# on 13 Aug 2026: input 'inputs' [batch,160000], output 'embedding' [batch,1536], real
# inference measured at ~156ms/clip (batch=16, 4 CPU threads) on a laptop CPU -- NOT the
# target embedded device, a reference point only. Peak-norm was confirmed to materially
# change the real embedding (mean abs delta 0.065), i.e. the v1 peak-norm bug was real,
# not theoretical, on the actual weights.
_ONNX_HF_URL = "https://huggingface.co/justinchuby/Perch-onnx/resolve/main/perch_v2_no_dft.onnx"

def _find_local_onnx():
    hits = (glob.glob('/kaggle/input/**/perch_v2_no_dft*.onnx', recursive=True)
            or glob.glob('/kaggle/input/**/perch_v2*.onnx', recursive=True))
    return hits[0] if hits else None

def _load_onnx_perch():
    import onnxruntime as ort
    path = _find_local_onnx()
    if path is None:
        # Falls back to a live download (~413MB) if nothing is attached as a Kaggle
        # input -- slow and wasteful on repeated runs. Attaching a Kaggle dataset
        # mirror (e.g. tuckerarrants/perch-v2-no-dft-onnx) is the fast path.
        import requests
        path = "/kaggle/working/perch_v2_no_dft.onnx"
        print(f"  no local ONNX Perch found; downloading from {_ONNX_HF_URL} ...")
        r = requests.get(_ONNX_HF_URL, stream=True, timeout=300)
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    so = ort.SessionOptions(); so.intra_op_num_threads = 4
    sess = ort.InferenceSession(path, sess_options=so, providers=["CPUExecutionProvider"])
    in_name = sess.get_inputs()[0].name
    out_names = [o.name for o in sess.get_outputs()]
    print(f"  ONNX Perch loaded from {path}  |  input='{in_name}'  outputs={out_names}")
    return sess, in_name, out_names

if PERCH_BACKEND == "onnx":
    _ort_sess, _ort_in, _ort_out_names = _load_onnx_perch()
    _HAS_INFER_TF = False; _SERVING = None; perch = None   # not used on this path
    print(f"  calling convention: onnxruntime (CPU)  |  peak-norm: {PERCH_PEAK_NORM}")
else:
    PERCH_HANDLE = _find_local_perch(PERCH_VERSION == "v2") or _remote_handle()
    print(f"Perch {PERCH_VERSION} source: {PERCH_HANDLE}")
    perch = hub.load(PERCH_HANDLE)

    # Which calling convention does THIS artefact expose? Detected, not assumed -- so a
    # mis-attached Kaggle input degrades to a clear message instead of a confusing crash.
    _HAS_INFER_TF = hasattr(perch, "infer_tf")
    _SERVING = None
    if not _HAS_INFER_TF:
        try:
            _SERVING = perch.signatures["serving_default"]
        except Exception as ex:
            raise RuntimeError(
                "Loaded Perch artefact exposes neither infer_tf nor a 'serving_default' "
                f"signature ({type(ex).__name__}). Check the attached Kaggle model."
            ) from ex
    print(f"  calling convention: {'infer_tf (Perch 1.x)' if _HAS_INFER_TF else 'serving_default (Perch 2.x)'}"
          f" | peak-norm: {PERCH_PEAK_NORM}")
_perch_batched = True

def _probe_embed_dim():
    try:
        return int(_perch_run(np.zeros((1, PERCH_LEN), dtype="float32")).shape[-1])
    except Exception as ex:
        print("  could not probe embedding dim, assuming 1280:", type(ex).__name__)
        return 1280

def to32k(y):
    return librosa.resample(np.asarray(y, dtype="float32"), orig_sr=SR, target_sr=PERCH_SR)

def _peak_norm(batch):
    """DC-remove and scale each window to PERCH_PEAK_NORM -- zoo_interface.normalize_audio."""
    if PERCH_PEAK_NORM is None: return batch
    b = np.asarray(batch, dtype="float32").copy()
    b -= np.mean(b, axis=-1, keepdims=True)
    pk = np.max(np.abs(b), axis=-1, keepdims=True)
    b = np.divide(b, pk, where=(pk > 0.0))
    return (b * PERCH_PEAK_NORM).astype("float32")

def _extract_embedding(out):
    """Perch 2.x -> dict with 'embedding' (pooled) plus 'spatial_embedding'/'frontend'/
    logit heads. Perch 1.x -> dict with 'embedding', or a bare (logits, embeddings) tuple."""
    if hasattr(out, "keys"):
        if "embedding" not in out:
            raise KeyError(f"no 'embedding' head in Perch output; got {sorted(out.keys())}")
        emb = out["embedding"]
    else:
        emb = out[1]
    return np.asarray(emb.numpy() if hasattr(emb, "numpy") else emb)

def _infer(batch):
    batch = _peak_norm(batch)
    if PERCH_BACKEND == "onnx":
        result = _ort_sess.run(None, {_ort_in: batch})
        out = dict(zip(_ort_out_names, result))   # verified real keys: embedding/
                                                   # spatial_embedding/spectrogram/label
    else:
        out = perch.infer_tf(batch) if _HAS_INFER_TF else _SERVING(inputs=batch)
    return _extract_embedding(out)

def _perch_run(waves):
    global _perch_batched
    n = len(waves)
    if n == 0: return np.zeros((0, EMBED_DIM if "EMBED_DIM" in globals() else 1280), dtype="float32")
    if _perch_batched:
        try:
            out = []
            for i in range(0, n, PERCH_BATCH):
                chunk = waves[i:i+PERCH_BATCH]; k = len(chunk)
                if k < PERCH_BATCH:
                    chunk = np.concatenate([chunk, np.zeros((PERCH_BATCH-k, PERCH_LEN), dtype="float32")])
                out.append(_infer(chunk)[:k])
            return np.concatenate(out)
        except Exception as ex:
            print("batched Perch inference failed, falling back to per-clip:", type(ex).__name__)
            _perch_batched = False
    return np.stack([_infer(w[np.newaxis, :])[0] for w in waves])

def perch_embed(waveforms):
    """Isolated clips -> (n, D). CENTRED zero-padding, Perch's documented convention."""
    ws = []
    for w in waveforms:
        w = to32k(w)
        if len(w) >= PERCH_LEN:
            w = w[:PERCH_LEN]
        else:
            pad = PERCH_LEN - len(w); lo = pad // 2
            w = np.pad(w, (lo, pad - lo))
        ws.append(w.astype("float32"))
    return _perch_run(np.stack(ws)) if ws else np.zeros((0, EMBED_DIM), dtype="float32")

def perch_embed_ctx(y32, centres_sec):
    """Real 5s windows cut from a 32kHz recording, centred on each event time."""
    ws = []
    for c in centres_sec:
        a = int(round(c * PERCH_SR)) - PERCH_LEN // 2
        a = max(0, min(a, max(0, len(y32) - PERCH_LEN)))
        w = y32[a:a + PERCH_LEN]
        if len(w) < PERCH_LEN:
            pad = PERCH_LEN - len(w); lo = pad // 2
            w = np.pad(w, (lo, pad - lo))
        ws.append(w.astype("float32"))
    return _perch_run(np.stack(ws)) if ws else np.zeros((0, EMBED_DIM), dtype="float32")

EMBED_DIM = _probe_embed_dim()
_DETECTED = {1280: "v1", 1536: "v2"}.get(EMBED_DIM, "unknown")
print(f"Perch embedding dim: {EMBED_DIM}  (detected {_DETECTED}, requested {PERCH_VERSION})")
if _DETECTED != PERCH_VERSION:
    # Wrong artefact attached is the likeliest cause, and it would otherwise produce
    # perfectly plausible numbers from the wrong model. Refuse rather than mislead.
    raise RuntimeError(
        f"Perch version mismatch: requested {PERCH_VERSION} but the loaded model emits "
        f"{EMBED_DIM}-d embeddings ({_DETECTED}). Loaded from: {PERCH_HANDLE}\n"
        f"  -> On Kaggle, attach the model variation matching PERCH_VERSION:\n"
        f"     v2: bird-vocalization-classifier/tensorFlow2/perch_v2 (GPU, v2) "
        f"or perch_v2_cpu (CPU, v1)\n"
        f"     v1: bird-vocalization-classifier/tensorFlow2/bird-vocalization-classifier (v8)")

# ------------------------------------------------- pre-registered target selection
_counts = meta.primary_label.value_counts()
_eligible = [sp for sp in _counts.index if _counts[sp] >= MIN_RECS]
if PRIMARY_TARGET not in _eligible:
    raise ValueError(f"{PRIMARY_TARGET} has only {_counts.get(PRIMARY_TARGET, 0)} recordings "
                     f"(need >= {MIN_RECS})")
if SINGLE_TARGET_COMPARISON:
    TARGET_LIST = [PRIMARY_TARGET]
else:
    TARGET_LIST = [PRIMARY_TARGET] + [sp for sp in _eligible if sp != PRIMARY_TARGET][:N_TARGETS - 1]
    if SMOKE_TEST:
        TARGET_LIST = TARGET_LIST[:2]
TARGET_SET = set(TARGET_LIST)     # <-- consumed by CELL 2 to exclude ALL targets from the
                                  #     encoder bank. v1 hard-coded "whbsho3" there, which
                                  #     silently trained the encoder on every OTHER target.
print(f"\nPRE-REGISTERED TARGETS (n={len(TARGET_LIST)}, smoke_test={SMOKE_TEST}, "
      f"single_target_comparison={SINGLE_TARGET_COMPARISON}) "
      f"-- copy this list into the logbook BEFORE running:")
for sp in TARGET_LIST:
    print(f"   {sp}  ({_counts[sp]} recordings)")

# ------------------------------------------------- global species cache (ONCE)
_cand = (meta[~meta.primary_label.isin(TARGET_SET)]['primary_label']
         .value_counts().head(N_CANDIDATES).index.tolist())
SPECIES_CACHE = {}
_t0 = time.time()
for sp in _cand:
    calls = []
    for f in meta[meta.primary_label == sp]['filename'].tolist()[:CALIB_PER_SPECIES]:
        try: calls += peak_windows(ld(os.path.join(AUD, f), 30), k=1)
        except Exception: pass
    if not calls: continue
    SPECIES_CACHE[sp] = {"calls": calls, "emb": l2n(perch_embed(calls)).mean(0)}
print(f"species cache: {len(SPECIES_CACHE)}/{len(_cand)} candidates embedded "
      f"in {time.time()-_t0:.0f}s (computed once, reused by every target)")

_decoy_pool_cache = {}
def _deep_decoy_pool(sp):
    if sp not in _decoy_pool_cache:
        pool = []
        for f in meta[meta.primary_label == sp]['filename'].tolist()[:DECOY_RECORDINGS]:
            try: pool += peak_windows(ld(os.path.join(AUD, f), 30), k=1)
            except Exception: pass
        _decoy_pool_cache[sp] = pool or SPECIES_CACHE[sp]["calls"]
    return _decoy_pool_cache[sp]

# ------------------------------------------------------------------ soundscape beds
bed_paths = sorted(glob.glob(os.path.join(SS, "*.ogg")))[:N_CALIB_BEDS + N_EVAL_BEDS]
all_beds = [ld(p, 240.0) for p in bed_paths]
calib_beds, eval_beds = all_beds[:N_CALIB_BEDS], all_beds[N_CALIB_BEDS:]
BED_SEC = 240.0
print(f"beds: {len(calib_beds)} calibration + {len(eval_beds)} evaluation (disjoint)")

# ------------------------------------------------------------ shot selection
def choose_shots(files, ratings, embs, n_shot, strategy, rng):
    """-> indices into `files`. Each strategy is a different answer to the open question
    Nolasco et al. sec.4.1 raise: which 5 examples should a deployed system be given?
      rated   : highest Xeno-Canto rating (ARGUS v1 behaviour -- best-quality audio)
      first   : first n in metadata order (DCASE Task 5 behaviour -- no curation)
      random  : uniform sample (control)
      diverse : greedy max-min distance in Perch space (maximum coverage of the
                call repertoire -- the thing the expert annotators said was missing)
    """
    n = len(files)
    if strategy == "rated":
        return list(np.argsort(-np.asarray(ratings, dtype=float))[:n_shot])
    if strategy == "first":
        return list(range(min(n_shot, n)))
    if strategy == "random":
        return list(rng.permutation(n)[:n_shot])
    if strategy == "diverse":
        E = l2n(embs)
        picked = [int(np.argmax(np.asarray(ratings, dtype=float)))]   # deterministic seed point
        while len(picked) < min(n_shot, n):
            d = np.min(1.0 - E @ E[picked].T, axis=1)   # cosine distance to nearest picked
            d[picked] = -np.inf
            picked.append(int(np.argmax(d)))
        return picked
    raise ValueError(f"unknown shot strategy: {strategy}")

# ------------------------------------------------------------ stage-2 verifiers
class PrototypicalProbe:
    """Prototype-distance classifier. Drop-in for LogisticRegression (fit/predict_proba).

    WHY: Perch 2.0's own benchmark (Table 3, BEANS) reports prototypical probing beating
    linear probing on DETECTION tasks by +0.078 mAP (0.426 -> 0.504). ARGUS stage 2 is a
    detection task, and currently uses a linear probe.

    K prototypes per class rather than one mean, following the ProtoPNet lineage Perch 2.0
    adopts (4 prototypes/class, prediction = max activation across them). This is also the
    direct answer to Nolasco et al. sec.4.1: a single averaged prototype "implies that the
    examples are in some sense all of one kind" -- false for a species with several call
    types, which is exactly ARGUS's failure mode.

    Scores are cosine similarities (inputs are L2-normalised upstream), turned into
    calibrated probabilities by a fitted temperature. Calibration matters here because the
    fusion rules multiply p*v -- an uncalibrated v that saturates to 0/1 becomes a hard
    veto and destroys ranking, which is the same trap that forced C=0.1 on the LogReg.
    """

    def __init__(self, n_prototypes=4, random_state=0, class_weight="balanced"):
        self.n_prototypes = n_prototypes
        self.random_state = random_state
        self.class_weight = class_weight

    def _protos(self, Xc):
        k = int(min(self.n_prototypes, len(Xc)))
        if k <= 1:
            return l2n(Xc.mean(0, keepdims=True))
        km = KMeans(n_clusters=k, n_init=10, random_state=self.random_state).fit(Xc)
        return l2n(km.cluster_centers_)

    def _scores(self, X):
        """-> (n, 2): max cosine similarity to each class's prototype set."""
        Xn = l2n(X)
        return np.stack([(Xn @ self.protos_[c].T).max(1) for c in (0, 1)], axis=1)

    @staticmethod
    def _softmax(z):
        z = z - z.max(axis=1, keepdims=True)
        e = np.exp(z)
        return e / e.sum(axis=1, keepdims=True)

    def _fit_temperature(self, S, y, w, eps=0.05):
        """Temperature by weighted log-loss with LABEL SMOOTHING.

        Smoothing is not cosmetic here. The calibration set separates almost perfectly in
        Perch space, so an unsmoothed objective drives T -> 0 and saturates predict_proba
        to hard 0/1. That turns v into a veto rather than a score, which destroys the
        ranking that AP and the p*v fusion rules depend on -- the identical failure mode
        that forced C=0.1 on the LogisticRegression verifier. With targets smoothed to
        1-eps the optimum is finite and the probabilities stay soft.
        """
        best_T, best_ll = 1.0, np.inf
        for T in np.geomspace(0.02, 3.0, 60):
            p_true = np.clip(self._softmax(S / T)[np.arange(len(y)), y], 1e-9, 1.0)
            p_false = np.clip(1.0 - p_true, 1e-9, 1.0)
            ll = float(-(w * ((1.0 - eps) * np.log(p_true)
                              + eps * np.log(p_false))).sum() / w.sum())
            if ll < best_ll:
                best_T, best_ll = float(T), ll
        return best_T

    def fit(self, X, y):
        X = np.asarray(X, dtype="float32"); y = np.asarray(y).astype(int)
        self.protos_ = {c: self._protos(X[y == c]) for c in (0, 1)}
        if self.class_weight == "balanced":
            cw = {c: len(y) / (2.0 * max(1, (y == c).sum())) for c in (0, 1)}
        else:
            cw = {0: 1.0, 1: 1.0}
        w = np.array([cw[int(v)] for v in y], dtype="float64")
        self.temperature_ = self._fit_temperature(self._scores(X), y, w)
        return self

    def predict_proba(self, X):
        return self._softmax(self._scores(X) / self.temperature_)

    def score(self, X, y):
        return float((self.predict_proba(X).argmax(1) == np.asarray(y).astype(int)).mean())


def _make_verifier():
    if VERIFIER == "logreg":
        # C deliberately small: 1280/1536-d embeddings with ~100 samples separate perfectly
        # at C=1, saturating predict_proba to 0/1 and turning v into a hard veto.
        return LogisticRegression(C=0.1, max_iter=2000, class_weight="balanced")
    if VERIFIER == "proto":
        return PrototypicalProbe(n_prototypes=4, random_state=0)
    raise ValueError(f"unknown VERIFIER: {VERIFIER}")


# ------------------------------------------------------------ per-species covariates
def _covariates(support_views, own_embs, decoy_sims):
    """Recorded so the difficulty-factor regression Nolasco et al. could not resolve
    becomes possible later. Free now; unreconstructable after the fact."""
    E = l2n(own_embs)
    if len(E) > 1:
        S = E @ E.T
        iu = np.triu_indices(len(E), k=1)
        stereotypy = float(S[iu].mean())        # high = calls resemble each other
    else:
        stereotypy = float('nan')
    cents, bws, durs = [], [], []
    for w in support_views:
        cents.append(float(np.mean(librosa.feature.spectral_centroid(y=w, sr=SR))))
        bws.append(float(np.mean(librosa.feature.spectral_bandwidth(y=w, sr=SR))))
        e = np.square(w, dtype=np.float64)
        thr = 0.1 * e.max() if e.max() > 0 else 0.0
        durs.append(float((e > thr).sum()) / SR)   # seconds above 10% of peak energy
    return dict(
        stereotypy=stereotypy,
        spectral_centroid_hz=float(np.mean(cents)),
        spectral_bandwidth_hz=float(np.mean(bws)),
        call_duration_s=float(np.mean(durs)),
        decoy_sim_top1=float(max(decoy_sims)) if decoy_sims else float('nan'),
        decoy_sim_mean=float(np.mean(decoy_sims)) if decoy_sims else float('nan'),
    )

# ------------------------------------------------------------ per-target builder
# Everything except the verifier fit is deterministic given (species, n_shot, strategy,
# seed) -- and it is nearly all Perch inference, the dominant cost. The ablation ladder
# rebuilds targets once per rung, so without this cache every rung would re-embed the same
# audio. Cached across rungs; only the (cheap) classifier is refitted.
_TARGET_CACHE = {}

def build_target(sp, n_shot=N_SHOT, strategy=SHOT_STRATEGY, seed=0, verbose=True,
                  decoy_mode="hard", recordist_disjoint=False):
    """Everything v1 did at module level for one hard-coded species, now per species.
    Returns a self-contained context dict consumed by CELL 2.

    decoy_mode selects WHICH species (out of every candidate ranked by Perch-embedding
    similarity to this target) become the N_DECOYS used at test time -- roadmap 3.3:
      'hard'   (default) -- the top N_DECOYS nearest. Unchanged behaviour; every result
                published before this flag existed used this mode.
      'medium' -- ranks N_DECOYS..2*N_DECOYS-1 (5th-8th nearest at N_DECOYS=4).
      'random' -- N_DECOYS drawn uniformly from the full non-target candidate pool,
                  seeded from this call's own `seed` so it's reproducible, not a
                  fresh draw on every call.
    Only what a run is SCORED against changes; the verifier is still trained against
    every non-decoy candidate as negatives regardless of mode (calib_neg_species below),
    so 'easier' decoys are a harder TEST, not a easier training signal.

    recordist_disjoint -- confound-closing run 3 (roadmap "owed" list): "did it learn the
    microphone, not the bird?" Default False reproduces every previously-published number
    byte-for-byte (held-out pool = every recording not used as a shot, regardless of who
    recorded it). True additionally excludes, from the held-out pool, any recording whose
    RECORDIST_COL matches one of the n_shot support recordings -- so if a species'
    apparent AUC were actually partly "recognise this recordist's mic/room", a
    recordist-disjoint held-out pool would show it as a drop. Can raise ValueError if a
    species has too few distinct recordists (the held-out pool empties out) -- that itself
    is worth logging, not a bug to route around."""
    # EMBED_BACKEND is part of the key for the same reason recordist_disjoint is: the cache
    # stores the fitted verifier's TRAINING MATRIX (c["X"]) and refits only the cheap
    # classifier on it. Without the backend in the key, switching to the distilled encoder
    # would silently refit on Perch's cached embeddings and report the result as distilled.
    ck = (sp, int(n_shot), strategy, int(seed), decoy_mode, bool(recordist_disjoint),
          EMBED_BACKEND)
    if ck in _TARGET_CACHE:
        c = _TARGET_CACHE[ck]
        clf = _make_verifier().fit(c["X"], c["yv"])
        cov = dict(c["cov"]); cov["clf_train_acc"] = float(clf.score(c["X"], c["yv"]))
        cov["verifier"] = VERIFIER
        if verbose:
            print(f"  [{sp}] cached build reused; verifier={VERIFIER} "
                  f"(train acc {cov['clf_train_acc']:.2f})")
        return dict(species=sp, support=c["support"], target_pool=c["target_pool"],
                    PROTO=c["PROTO"], DECOYS=c["DECOYS"], decoy_pools=c["decoy_pools"],
                    perch_clf=clf, covariates=cov, neg_calls=c["neg_calls"])
    rng = np.random.default_rng(seed)
    rows = meta[meta.primary_label == sp]
    files   = rows['filename'].tolist()
    ratings = rows['rating'].tolist() if 'rating' in rows else [0.0] * len(files)
    if recordist_disjoint and RECORDIST_COL not in rows:
        raise KeyError(f"recordist_disjoint=True but {RECORDIST_COL!r} is not a column "
                        f"of train_metadata.csv; have {rows.columns.tolist()}")
    authors = (rows[RECORDIST_COL].tolist() if recordist_disjoint
               else [None] * len(files))
    if len(files) < n_shot + 2:
        raise ValueError(f"not enough recordings for {sp}: {len(files)}")

    raw = [ld(os.path.join(AUD, f), 30) for f in files]
    # 'diverse' needs one embedding per recording to pick a spread; others don't.
    rec_embs = (perch_embed([peak_windows(y, k=1)[0] for y in raw])
                if strategy == "diverse" else np.zeros((len(raw), 1)))

    shot_idx = choose_shots(files, ratings, rec_embs, n_shot, strategy, rng)
    shot_set = set(int(i) for i in shot_idx)

    support = build_support_views([raw[i] for i in shot_idx])
    # Held-out pool = every recording NOT used as a shot. v1 sliced files[N_SHOT:],
    # which under any non-'first' strategy would have leaked support into evaluation.
    excluded_idx = set(shot_set)
    if recordist_disjoint:
        # pandas gives every missing value in a string column the SAME NaN singleton, and
        # `a in shot_authors` does an identity check before equality -- so a NaN-authored
        # shot correctly excludes every other NaN-authored recording as one shared
        # pseudo-recordist. This is conservative (over-exclusion, shrinking the held-out
        # pool), not a leak -- verified 28 Aug 2026 against the real pandas behaviour after
        # an audit initially suspected the opposite (NaN != NaN silently under-excluding).
        # It fails safe, so the only thing worth doing is making it visible.
        n_nan = sum(1 for a in authors if a is not None and a != a)
        if n_nan and verbose:
            print(f"  [{sp}] {n_nan}/{len(authors)} recordings have a missing author -- "
                  f"treated as one shared pseudo-recordist for recordist_disjoint")
        shot_authors = {authors[i] for i in shot_set if authors[i] is not None}
        excluded_idx |= {i for i, a in enumerate(authors) if a in shot_authors}
    target_pool = [w for i, y in enumerate(raw) if i not in excluded_idx
                   for w in peak_windows(y, k=1)]
    if not target_pool:
        extra = (f" ({len(excluded_idx) - len(shot_set)} extra recordings excluded for "
                 f"sharing a recordist with a shot)" if recordist_disjoint else "")
        raise ValueError(f"{sp}: no held-out recordings left after taking {n_shot} "
                          f"shots{extra}")

    own_embs = perch_embed(support)
    PROTO = l2n(own_embs).mean(0); PROTO /= np.linalg.norm(PROTO) + 1e-9

    pool_sp = {k: v for k, v in SPECIES_CACHE.items() if k != sp}
    ranked  = sorted(pool_sp, key=lambda o: -cos(pool_sp[o]["emb"], PROTO))
    if decoy_mode == "hard":
        DECOYS = ranked[:N_DECOYS]
    elif decoy_mode == "medium":
        DECOYS = ranked[N_DECOYS:2 * N_DECOYS]
        if len(DECOYS) < N_DECOYS:
            raise ValueError(f"{sp}: only {len(ranked)} candidates ranked, need "
                              f"{2*N_DECOYS} for decoy_mode='medium'")
    elif decoy_mode == "random":
        # Seeded from this call's `seed`, offset so it never coincides with any other
        # rng drawn from the same seed in this function (shot choice, calibration).
        DECOYS = list(np.random.default_rng(90_000 + seed).choice(
            ranked, size=N_DECOYS, replace=False))
    else:
        raise ValueError(f"unknown decoy_mode: {decoy_mode!r} (want hard|medium|random)")
    decoy_sims  = [cos(pool_sp[o]["emb"], PROTO) for o in DECOYS]
    decoy_pools = {o: _deep_decoy_pool(o) for o in DECOYS}
    calib_neg_species = [o for o in ranked if o not in DECOYS]

    # ---- domain-matched perch_clf (identical protocol to v1, per target) ----
    neg_calls = [w for o in calib_neg_species for w in pool_sp[o]["calls"]]
    rng_cal = np.random.default_rng(11 + seed)
    X_parts, y_all = [], []
    for bed in calib_beds:
        y_bed = bed.copy(); spans, centres, labels = [], [], []
        plan = [(1, support)] * 12 + [(0, neg_calls)] * 12
        plan = [plan[i] for i in rng_cal.permutation(len(plan))]
        for i, (lab, pl) in enumerate(plan):
            c = place(y_bed, pl, rng_cal, spans, SNR_LEVELS[i % len(SNR_LEVELS)])
            if c is None: continue
            centres.append(c + float(rng_cal.uniform(-0.25, 0.25))); labels.append(lab)
        for _ in range(8):
            pos = int(rng_cal.integers(0, len(y_bed) - CLEN))
            if any(pos < e and pos + CLEN > s for s, e in spans): continue
            centres.append((pos + CLEN/2) / SR); labels.append(0)
        X_parts.append(verifier_embed_ctx(y_bed, centres)); y_all += labels

    X = l2n(np.concatenate(X_parts)); yv = np.array(y_all)
    clf = _make_verifier().fit(X, yv)

    cov = _covariates(support, own_embs, decoy_sims)
    cov.update(species=sp, n_recordings=len(files), n_shot=n_shot,
               shot_strategy=strategy, n_heldout_calls=len(target_pool),
               clf_train_acc=float(clf.score(X, yv)),
               decoy_mode=decoy_mode,
               recordist_disjoint=bool(recordist_disjoint),
               n_distinct_recordists=(len(set(authors)) if recordist_disjoint else -1),
               # provenance: a results CSV that does not say which Perch produced it
               # is not comparable to any other run.
               perch_version=PERCH_VERSION, perch_embed_dim=EMBED_DIM,
               perch_peak_norm=(PERCH_PEAK_NORM if PERCH_PEAK_NORM is not None else 0.0),
               perch_backend=PERCH_BACKEND,
               verifier=VERIFIER)

    if verbose:
        rd_note = (f"  distinct_recordists={cov['n_distinct_recordists']}"
                   if recordist_disjoint else "")
        print(f"  [{sp}] shots={strategy}({n_shot})  heldout={len(target_pool)} calls  "
              f"decoys={DECOYS} ({decoy_mode})  top-sim={cov['decoy_sim_top1']:.3f}  "
              f"stereotypy={cov['stereotypy']:.3f}  verifier={VERIFIER}{rd_note}")

    _TARGET_CACHE[ck] = dict(support=support, target_pool=target_pool, PROTO=PROTO,
                             DECOYS=DECOYS, decoy_pools=decoy_pools, X=X, yv=yv,
                             neg_calls=neg_calls,
                             cov={k: v for k, v in cov.items() if k != "clf_train_acc"})
    return dict(species=sp, support=support, target_pool=target_pool, PROTO=PROTO,
                DECOYS=DECOYS, decoy_pools=decoy_pools, perch_clf=clf, covariates=cov,
                neg_calls=neg_calls)

print("\nCELL 1 v2 ready. build_target(sp) is the entry point; TARGET_SET is exported "
      "for the encoder-bank exclusion in CELL 2.")


In [ ]:
# ===== CELL 2 v2 / TWO-STAGE across N species =====
# Needs CELL 1 v2 (arguswala_setup_v2.py).
#
# WHAT CHANGED vs v1:
#
#  (1) THE BUG. v1 line 216 read:
#          sps = [s for s in meta.primary_label.unique() if s != "whbsho3"]
#      The target was hard-coded while CELL 1 defined it as a variable. The moment a
#      second target is evaluated, that line puts the new target INTO the encoder's
#      training bank -- the encoder is pre-trained on the species it must never have
#      seen, every number inflates, and nothing crashes. Now excludes all of TARGET_SET,
#      with an assertion so it can never regress silently.
#
#  (2) ONE ENCODER, MANY TARGETS. Legitimate because no target is in the bank, and it
#      turns O(species x encoder-trainings) into O(1).
#
#  (3) PURE-NEGATIVE CONTROL (neg_pass). Nothing in v1 ever ran on a recording with zero
#      target calls -- in deployment that is the common case, and false alarms per hour
#      on clean audio is the first number a conservation biologist asks for.
#
#  (4) FEATURE TOGGLE. Liang et al. (DCASE 2024 baseline) Table III: PCEN gives the best
#      PRECISION of any front-end tested (68.0%) while log-Mel gives the best F1 (63.7 vs
#      60.0). v1 assumed PCEN. Now it is a measured choice.
#
#  (5) COVARIATES + per-species results written to CSV, so the difficulty-factor analysis
#      Nolasco et al. attempted and failed can actually be run afterwards.
import numpy as np, os, glob, copy, time, librosa, torch, torch.nn as nn, torch.optim as optim
import pandas as pd

SR, N_FFT, HOP, N_MELS = 22050, 1024, 256, 128   # identical to the DCASE 2024 baseline
                                                 # (22.05kHz / 1024 / 256 / 128) -- keep it
                                                 # that way, it is what makes a DCASE run cheap
SEG_SEC = 1.0; SEG_FR = int(np.ceil(SEG_SEC*SR/HOP)); CLEN = int(SEG_SEC*SR); EMB = 128
FT_STEPS, FT_LR, FT_BATCH = 250, 1e-4, 16
N_INJECT, N_NEG, INJ_SNR = 45, 200, (-6, 12)
CAND_FLOOR, STRIDE, IOU_THR = 0.10, max(1, SEG_FR//3), 0.30
MAX_EVENT_SEC = 10.0
CALLS_PER_BED = 12

FEATURE = "pcen"          # 'pcen' | 'logmel'
SEEDS   = (7, 8, 9)       # raise to >=10 once the timing measurement says it fits
ENCODER_SEEDS = (0,)  # DEMO: one encoder is enough; research run used (0, 1, 2) # >1 entry measures encoder-training variance (roadmap 3.4).
                          # Widened 14 Aug 2026 after the first real run: whbsho3 stage-1
                          # AUC came back 0.517 here vs. the earlier single-target baseline
                          # of 0.459. This tests ONE candidate cause -- training-run
                          # stochasticity on a FIXED bank (build_banks() always uses
                          # seed=0 regardless of this tuple, so bank COMPOSITION doesn't
                          # vary here). A separate single-target comparison run is still
                          # the only way to test the other candidate cause (bank
                          # composition differs because this run excludes 2 species from
                          # the encoder bank, the original baseline only excluded 1).

# 18 Aug 2026: a Kaggle GPU-quota cutoff killed a run mid-way through CELL 4 after
# CELL 2's 5.5-hour species loop had already completed and its CSV was safely
# downloaded. Restarting the whole notebook to reach CELL 4 again would re-pay that
# 5.5 hours for a result already in hand. SKIP_MAIN_LOOP=True runs bank-building and
# encoder training (~10-20 min, everything CELL 4 actually needs) and stops there --
# `res`/`wide`/the printed report below never run, and never need to. CELL 4's own
# checkpointing (RESUME=True) then picks up the ladder at the first unfinished config.
# Leave False for a normal full run.
SKIP_MAIN_LOOP = True

# ---------------------------------------------------------------- ablation ladder
# Stage-1 rungs. Both default OFF so rung 0 reproduces the original pipeline exactly.
# (Stage-2 rung lives in CELL 1 as VERIFIER = 'logreg' | 'proto'.)
USE_TIMEFILTER = False    # rung 2: TimeFilterAug on the support during fine-tuning
USE_HARD_NEG   = False    # rung 3: hard-negative mining in the fine-tuning loop
HARD_NEG_KEEP  = 0.25     # fraction of candidate negatives retained as "hard"

# rung 5 (CELL 5, confound-closing run 2 -- roadmap "owed" list): stage-1's per-target
# fine-tuning has always drawn its negatives as arbitrary random crops of the bed
# (whatever silence/wind/distant noise happens to be there) -- never a labelled OTHER
# SPECIES call. The stage-2 Perch verifier already trains against real other-species
# calls (calib_neg_species in build_target); stage 1 never has. Default OFF reproduces
# every previously-published number byte-for-byte. When True, stage1() injects real
# calls from ctx['neg_calls'] (the SAME non-decoy species pool the verifier uses) into a
# copy of the clean recording and crops THOSE positions as negatives -- but N_NEG=200
# 1s calls with 0.4s gaps need >=280s in a 240s bed, which is geometrically impossible
# (measured: rejection-sampling places ~126 of the 200 target, min 122/max 130 over 10
# draws), so on THIS side the "instead of arbitrary background" framing overclaims --
# ~74 of the ~200 negatives are still background top-up on every call (see the top-up
# branch in stage1() below). What this rung actually tests is "~126 species-matched +
# ~74 background" vs "~200 background", not a clean substitution. Restated honestly in
# both places 28 Aug 2026, after an audit caught the original wording.
USE_SPECIES_NEG = False

# CELL 5, confound-closing run 1: how far a sliding Perch-embedding + verifier scan
# steps between windows when scoring a recording with NO stage-1 CNN in the loop at all
# (detect_perch_only below). Perch inference is the pipeline's dominant per-call cost, so
# this trades runtime against localisation precision; narrow it only after the first
# timing print says the wider stride is affordable.
PERCH_ALONE_STRIDE_S = 0.5

def timefilteraug(x, rng, m_range=(3, 6), lo_db=-6.0, hi_db=8.0):
    """TimeFilterAug -- Zou et al. 2024 (DCASE 2023 Task 5, 1st place, 63.8% F), eq. 6.

    Motivation, in their words: the five shots "are typically clear, whereas the query set
    for predictions often contains interference noise, predominantly from far-field sound
    and background impulse noise." This simulates that by applying a random piecewise-
    LINEAR gain envelope across TIME (not a mask, and not across frequency):

      partition the window into m in [3,6] segments; draw a breakpoint gain g_i ~ U[0,1]
      per boundary; within each segment interpolate linearly between adjacent g's; map
      that ramp onto [lo_db, hi_db] = [-6, +8] dB; apply.

    Their front-end is PCEN at 22050 Hz / n_fft 1024 / hop 256 -- identical to ARGUS, so
    the dB bounds transfer directly rather than needing rescaling.
    """
    T = x.shape[1]
    m = int(rng.integers(m_range[0], m_range[1] + 1))
    if T < m + 1:
        return x
    cuts  = np.sort(rng.choice(np.arange(1, T), size=m - 1, replace=False))
    edges = np.concatenate([[0], cuts, [T]])
    g = rng.random(m + 1)
    alpha = np.concatenate([
        np.linspace(g[i], g[i + 1], int(edges[i + 1] - edges[i]), endpoint=False)
        for i in range(m)
    ])
    beta = (lo_db + (hi_db - lo_db) * alpha).astype("float32")   # per-frame gain in dB
    if FEATURE == "logmel":
        # amplitude_to_db output is ALREADY logarithmic -> a dB gain is additive.
        return x + beta[np.newaxis, :]
    # PCEN is a linear-domain magnitude -> a dB gain is multiplicative.
    return x * (10.0 ** (beta / 20.0))[np.newaxis, :].astype("float32")

def aug_pos(x, rng):
    """Augmentation applied to positives (the injected support) during fine-tuning."""
    x = specaug(x, rng)
    if USE_TIMEFILTER:
        x = timefilteraug(x, rng)
    return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:", device)

def rms(x): return float(np.sqrt(np.mean(np.square(x, dtype=np.float64)) + 1e-12))
def ld(p, dur=None):
    y, _ = librosa.load(p, sr=SR, mono=True, duration=dur); return y.astype("float32")

FEATURES = ("pcen",)  # DEMO: log-Mel encoder not needed; research run used ("pcen", "logmel")   # every front-end an encoder is trained for

def feat(y, feature=None):
    """Front-end. PCEN vs log-Mel is an experiment, not an assumption.

    `feature=None` reads the FEATURE global -- that is what makes the ladder able to
    switch front-ends by rebinding a global. Pass it explicitly when featurising for a
    front-end other than the currently-selected one (e.g. building both encoder banks).
    """
    feature = feature or FEATURE
    m = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP,
                                       n_mels=N_MELS, power=1.0)
    if feature == "pcen":
        return librosa.pcen(m * (2**31), sr=SR, hop_length=HOP).astype("float32")
    if feature == "logmel":
        return librosa.amplitude_to_db(m, ref=np.max).astype("float32")
    raise ValueError(f"unknown feature: {feature}")

def crop(f, c):
    a = c - SEG_FR//2; p = np.full((N_MELS, SEG_FR), f.min(), dtype="float32")
    lo, hi = max(0, a), min(f.shape[1], a+SEG_FR)
    if hi > lo: p[:, (lo-a):(lo-a)+(hi-lo)] = f[:, lo:hi]
    return p
def loud_off(y, n=CLEN):
    if len(y) <= n: return 0
    e = np.square(y, dtype=np.float64); cs = np.concatenate([[0.0], np.cumsum(e)])
    return int(np.argmax(cs[n:]-cs[:-n]))
def specaug(x, rng):
    if rng.random() >= 0.6: return x
    x = x.copy(); fl = x.min()
    for _ in range(2):
        k = int(rng.integers(0, 21))
        if k: a = int(rng.integers(0, max(1, N_MELS-k))); x[a:a+k, :] = fl
        k = int(rng.integers(0, 11))
        if k: a = int(rng.integers(0, max(1, SEG_FR-k))); x[:, a:a+k] = fl
    return x
def inject(bed, calls, n, snr, rng, gap_s=0.4):
    y = bed.astype("float32").copy(); L = len(y); gap = int(gap_s*SR); sp = []
    if L <= CLEN: return y, sp
    for _ in range(n*60):
        if len(sp) >= n: break
        pos = int(rng.integers(0, L-CLEN))
        if any(pos < e+gap and pos+CLEN > s-gap for s, e in sp): continue
        c = calls[int(rng.integers(len(calls)))][:CLEN]
        if len(c) < CLEN: c = np.pad(c, (0, CLEN-len(c)))
        g = rms(y[pos:pos+CLEN])*(10**(float(rng.uniform(*snr))/20))/(rms(c)+1e-9)
        y[pos:pos+CLEN] += c.astype("float32")*g; sp.append((pos, pos+CLEN))
    return y, sorted(sp)
def iou(a, b):
    it = max(0.0, min(a[1], b[1]) - max(a[0], b[0]))
    u = (a[1]-a[0]) + (b[1]-b[0]) - it
    return it/u if u > 0 else 0.0

class Enc(nn.Module):
    def __init__(s, emb=EMB):
        super().__init__()
        def B(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False),
                                          nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        s.e = nn.Sequential(B(1,128), B(128,128), B(128,128), B(128,emb)); s.p = nn.AdaptiveAvgPool2d(1)
    def forward(s, x): return s.p(s.e(x.unsqueeze(1))).flatten(1)

def train_enc(bank, rng, episodes=1000, nway=6, k=4, q=4):
    m = Enc().to(device); opt = optim.Adam(m.parameters(), 1e-3); ce = nn.CrossEntropyLoss()
    cl = list(bank); m.train()
    for ep in range(episodes):
        ch = rng.choice(cl, size=min(nway, len(cl)), replace=False)
        sx, sy, qx, qy = [], [], [], []
        for lab, c in enumerate(ch):
            idx = rng.permutation(len(bank[c]))[:k+q]
            for j, i in enumerate(idx):
                (sx if j < k else qx).append(torch.tensor(specaug(bank[c][i], rng)))
                (sy if j < k else qy).append(lab)
        sx = torch.stack(sx).to(device); qx = torch.stack(qx).to(device)
        sy = torch.tensor(sy).to(device); qy = torch.tensor(qy).to(device)
        se, qe = m(sx), m(qx)
        pr = torch.stack([se[sy == c].mean(0) for c in torch.unique(sy)])
        loss = ce(-torch.cdist(qe, pr), qy); opt.zero_grad(); loss.backward(); opt.step()
        if (ep+1) % 250 == 0: print(f"    enc ep {ep+1}/{episodes} loss {loss.item():.3f}")
    return m

def stitch(frames, probs):
    ab = probs >= CAND_FLOOR; ev = []; i = 0
    while i < len(frames):
        if not ab[i]: i += 1; continue
        j = i
        while j+1 < len(frames) and ab[j+1] and frames[j+1]-frames[j] <= STRIDE*1.5: j += 1
        pk = i + int(np.argmax(probs[i:j+1]))
        s = max(0.0, frames[i]*HOP/SR - SEG_SEC/2); e = frames[j]*HOP/SR + SEG_SEC/2
        if e - s > MAX_EVENT_SEC:
            c = frames[pk]*HOP/SR; s, e = max(0.0, c-SEG_SEC/2), c+SEG_SEC/2
        ev.append((s, e, float(probs[pk]))); i = j+1
    return ev

def stitch_time(times, probs, floor=CAND_FLOOR, gap_s=None, max_event_sec=MAX_EVENT_SEC,
                 seg_sec=SEG_SEC):
    """Same merge-adjacent-above-floor logic as stitch(), but for detectors that never
    build a mel spectrogram (CELL 5's Perch-alone-as-detector), so positions are already
    in seconds rather than STRIDE-spaced spectrogram frame indices. `times` must be sorted
    ascending. gap_s defaults to a spacing consistent with stitch()'s STRIDE*1.5 in frames,
    converted to seconds."""
    gap_s = gap_s if gap_s is not None else (STRIDE * HOP / SR) * 1.5
    times = np.asarray(times, dtype="float64")
    ab = np.asarray(probs) >= floor; ev = []; i = 0
    while i < len(times):
        if not ab[i]: i += 1; continue
        j = i
        while j+1 < len(times) and ab[j+1] and times[j+1]-times[j] <= gap_s: j += 1
        pk = i + int(np.argmax(probs[i:j+1]))
        s = max(0.0, times[i] - seg_sec/2); e = times[j] + seg_sec/2
        if e - s > max_event_sec:
            c = times[pk]; s, e = max(0.0, c - seg_sec/2), c + seg_sec/2
        ev.append((s, e, float(probs[pk]))); i = j+1
    return ev

def stage1(enc, rec, sup, rng, neg_species_calls=None):
    n_inj = max(8, int(round(N_INJECT * len(rec) / (240.0 * SR))))
    aw, sp = inject(rec, sup, n_inj, INJ_SNR, rng)
    af, sf = feat(aw), feat(rec)
    pc = [((a+b)//2)//HOP for a, b in sp]
    if not pc: return []
    pos = [crop(af, c) for c in pc]
    g = SEG_FR; hi = af.shape[1]-g

    def _background_neg():
        out = []
        for _ in range(N_NEG*40):
            if len(out) >= N_NEG: break
            f = int(rng.integers(g, max(g+1, hi)))
            if all(abs(f-p) > SEG_FR for p in pc): out.append(crop(af, f))
        return out

    if USE_SPECIES_NEG and neg_species_calls:
        # Inject real OTHER-SPECIES calls into a FRESH copy of the clean recording (not
        # `aw`, which already carries the positive injections -- a separate copy means
        # positive and negative spans can never collide or corrupt each other) and crop
        # those exact centres as negatives. N_NEG=200 1s calls at 0.4s gaps cannot all
        # fit in a 240s bed (needs >=280s) -- inject() places ~126 of them, and the
        # len(neg)<N_NEG branch below tops up the rest with background. This arm is
        # therefore "~126 species-matched + ~74 background", not a clean 200-call
        # substitution -- see USE_SPECIES_NEG's definition above for the real numbers.
        nw, nsp = inject(rec, neg_species_calls, N_NEG, INJ_SNR, rng)
        nf = feat(nw)
        nc = [((a+b)//2)//HOP for a, b in nsp]
        neg = [crop(nf, c) for c in nc if g <= c <= nf.shape[1]-g]
        if len(neg) < N_NEG:
            # short recording / crowded bed couldn't place enough species-matched calls --
            # top up with background rather than silently training on fewer negatives.
            neg += _background_neg()
    else:
        neg = _background_neg()
    if not neg:
        neg = [np.full((N_MELS, SEG_FR), af.min(), dtype="float32")]
    m = copy.deepcopy(enc).to(device); h = nn.Linear(EMB, 2).to(device)
    opt = optim.Adam(list(m.parameters())+list(h.parameters()), FT_LR)
    ce = nn.CrossEntropyLoss(); m.train(); h.train(); hf = FT_BATCH//2

    def _train(steps, neg_pool):
        for _ in range(steps):
            xs  = [aug_pos(pos[int(rng.integers(len(pos)))], rng) for _ in range(hf)]
            xs += [neg_pool[int(rng.integers(len(neg_pool)))] for _ in range(hf)]
            xb = torch.tensor(np.stack(xs)).to(device)
            yb = torch.tensor([1]*hf+[0]*hf).to(device)
            loss = ce(h(m(xb)), yb); opt.zero_grad(); loss.backward(); opt.step()

    def _score(items):
        m.eval(); h.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(items), 256):
                b = torch.tensor(np.stack(items[i:i+256])).to(device)
                out.append(torch.softmax(h(m(b)), 1)[:, 1].cpu().numpy())
        m.train(); h.train()
        return np.concatenate(out)

    if USE_HARD_NEG and len(neg) >= 8:
        # Negative choice is named as decisive twice in the literature: Nolasco et al.
        # sec.4.2 ("prototype-based meta-learning works well when taking care about ...
        # the choice of negative examples") and Liang et al., who measure +5.31 F1 from
        # negative hard sampling. Random background windows are mostly trivial negatives
        # -- silence and wind -- so half the budget is spent learning nothing. Train
        # briefly, find the background the model currently CONFUSES with the target, and
        # spend the rest of the budget there.
        half = FT_STEPS // 2
        _train(half, neg)
        k = max(4, int(round(len(neg) * HARD_NEG_KEEP)))
        hard = [neg[i] for i in np.argsort(-_score(neg))[:k]]
        _train(FT_STEPS - half, hard)
    else:
        _train(FT_STEPS, neg)
    m.eval(); h.eval()
    fr = list(range(g, max(g+1, sf.shape[1]-g), STRIDE)); pb = np.empty(len(fr), dtype="float32")
    with torch.no_grad():
        for i in range(0, len(fr), 256):
            b = torch.tensor(np.stack([crop(sf, f) for f in fr[i:i+256]])).to(device)
            p = torch.softmax(h(m(b)), 1)[:, 1]; pb[i:i+len(p)] = p.cpu().numpy()
    return stitch(fr, pb)

def detect_pv(enc, ctx, rec, rng):
    """-> [(start, end, p, v)]. Target context is now explicit, not a global."""
    ev = stage1(enc, rec, ctx["support"], rng, neg_species_calls=ctx.get("neg_calls"))
    if not ev: return []
    centres = [(s + e) / 2 for s, e, _ in ev]
    # verifier_embed_ctx (CELL 1) dispatches on EMBED_BACKEND. build_target() FITS
    # ctx["perch_clf"] through the same function, so the classifier is always scored in the
    # space it was fit in -- under EMBED_BACKEND="distilled" no Perch inference runs here.
    emb = l2n(verifier_embed_ctx(rec, centres))
    v = ctx["perch_clf"].predict_proba(emb)[:, 1]
    return [(s, e, p, float(vi)) for (s, e, p), vi in zip(ev, v)]

def detect_perch_only(enc, ctx, rec, rng, stride_s=None):
    """-> [(start, end, p, v)], p := v. CELL 5, confound-closing run 1: "what did the CNN
    contribute?" No stage-1 encoder, no mel front-end, no fine-tuning loop anywhere in
    this path -- Perch embeddings are computed directly on a sliding window across the
    raw recording and scored with the SAME calibrated verifier (ctx['perch_clf']) every
    other detector in this file uses, then merged into events with stitch_time(). `enc`
    and `rng` are accepted only for call-site parity with detect_pv (so eval_pass /
    spec_pass / neg_pass can swap detectors via one `detect_fn` parameter) -- this path
    is deterministic given `rec`, so both are unused. p is set equal to v so the FUSIONS
    machinery still works mechanically, but only the "perch only (v)" fusion is a
    meaningful score here -- any p-dependent fusion rule applied to this detector's
    output is measuring nothing, since p carries no independent information."""
    stride_s = stride_s if stride_s is not None else PERCH_ALONE_STRIDE_S
    L = len(rec) / SR
    times = ([L / 2.0] if L <= SEG_SEC else
             list(np.arange(SEG_SEC / 2.0, L - SEG_SEC / 2.0 + 1e-9, stride_s)))
    if not times: return []
    emb = l2n(perch_embed_ctx(to32k(rec), times))
    v = ctx["perch_clf"].predict_proba(emb)[:, 1]
    # max_event_sec=SEG_SEC and an explicit gap_s are BOTH load-bearing; without them this
    # detector is at serious risk of scoring zero true positives in its normal operating
    # range, and failing silently in the direction of our own pre-registered prediction.
    # Geometry: Perch's window is 5 s, so a 1 s call sits inside it for ~11 consecutive
    # 0.5 s-strided positions, and if all of them clear CAND_FLOOR, stitch_time merges them
    # (0.5 < the default gap of 0.505 s) into one event spanning 0.5(R-1)+1 = 6 s. Truth
    # spans are 1 s, so IoU = 1/6 = 0.17, under IOU_THR=0.30 -- no match. Every merged-run
    # length R in [6,19] is unmatchable this way, and the default MAX_EVENT_SEC=10 only
    # rescues R>=20 (truncated to the peak window). This is a geometric hazard, not a
    # proven outcome -- whether real above-floor runs actually land in [6,19] depends on
    # the empirical density of the score field, which was never measured before this fix
    # landed (found by audit 28 Aug 2026, before Cell 5 ever ran). If it did land there
    # for most calls, the failure mode is silent and severe: near-zero true positives push
    # thr toward 1.0, drive FA/h toward 0 and AUC toward the 0.500 all-ties value, which
    # verdict() would report as "Perch alone is WORSE -- stage-1's localisation is doing
    # real work": our own pre-registered prediction, confirmed by an artefact rather than
    # a result. Capping at SEG_SEC removes the hazard rather than characterising it: it
    # collapses every run to a 1 s span at its peak window (R=1 is already 1 s and passes
    # through untouched), which is also the honest semantic -- a 5 s window cannot localise
    # better than its argmax. gap_s is passed explicitly so widening stride_s past 0.505 s
    # cannot silently flip merging off. Found by audit 28 Aug 2026, before this ever ran.
    ev = stitch_time(np.asarray(times), v, gap_s=max(stride_s * 1.5, 1e-6),
                     max_event_sec=SEG_SEC)
    return [(s, e, vi, vi) for s, e, vi in ev]

FUSIONS = {
    "stage-1 only  (p)":  lambda p, v: p,
    "perch only    (v)":  lambda p, v: v,
    "p * v":              lambda p, v: p * v,
    "p * sqrt(v)":        lambda p, v: p * np.sqrt(v),
    "sqrt(p) * v":        lambda p, v: np.sqrt(p) * v,
    "geo mean sqrt(p*v)": lambda p, v: np.sqrt(p * v),
}

# ------------------------------------------------------------------- metrics
def match(events, truth, iou_thresh):
    pairs = sorted(((iou(e[:2], t), i, j) for i, e in enumerate(events)
                    for j, t in enumerate(truth)), reverse=True)
    used_e, used_t = set(), set()
    for score, i, j in pairs:
        if score <= iou_thresh: break
        if i in used_e or j in used_t: continue
        used_e.add(i); used_t.add(j)
    return [(e[2], i in used_e) for i, e in enumerate(events)]

def match_truth(events, truth, iou_thresh):
    pairs = sorted(((iou(e[:2], t), i, j) for i, e in enumerate(events)
                    for j, t in enumerate(truth)), reverse=True)
    ue, ut, out = set(), set(), [0.0] * len(truth)
    for score, i, j in pairs:
        if score <= iou_thresh: break
        if i in ue or j in ut: continue
        ue.add(i); ut.add(j); out[j] = events[i][2]
    return out

def fp_per_tp(scored, n_truth, target_recall=0.5):
    arr = sorted(scored, key=lambda x: -x[0]); tp = fp = 0
    for score, hit in arr:
        if hit: tp += 1
        else:   fp += 1
        if tp / n_truth >= target_recall:
            return fp / tp if tp else float('inf')
    return float('inf')

def summarise(scored, n_truth):
    if not scored or not n_truth:
        return dict(ap=0.0, f1=0.0, p=0.0, r=0.0, thr=1.0, thr_r90=1.0, n=len(scored))
    arr = sorted(scored, key=lambda x: -x[0])
    tp = fp = 0; ap = 0.0; prev_r = 0.0; best = (0.0, 0.0, 0.0, 1.0); thr_r90 = None
    for score, hit in arr:
        if hit: tp += 1
        else:   fp += 1
        pr = tp / (tp + fp); rc = tp / n_truth
        ap += pr * (rc - prev_r); prev_r = rc
        f1 = 2*pr*rc/(pr+rc) if pr + rc else 0.0
        if f1 > best[0]: best = (f1, pr, rc, score)
        # Recall-first operating point (N4): the HIGHEST threshold still achieving >=90%
        # recall, i.e. best precision subject to a recall floor. Scores descend, so the
        # first crossing is that threshold. Setting it to the score just BEFORE the
        # crossing (the earlier bug) yields a point that never actually reaches 90%.
        if thr_r90 is None and rc >= 0.90: thr_r90 = score
    if thr_r90 is None: thr_r90 = arr[-1][0]   # 90% unreachable -> loosest available
    return dict(ap=ap, f1=best[0], p=best[1], r=best[2], thr=best[3],
                thr_r90=thr_r90, n=len(scored))

def _auc(pos, neg):
    """P(random target scores above random decoy), ties 0.5. Threshold-FREE.
    Same metric family BirdCLEF 2024 adopted as its official score (macro ROC-AUC):
    0.50 = cannot tell target from decoy, 1.00 = perfect species discrimination."""
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    if not len(pos) or not len(neg): return float('nan')
    gt = float((pos[:, None] > neg[None, :]).sum())
    eq = float((pos[:, None] == neg[None, :]).sum())
    return (gt + 0.5 * eq) / (len(pos) * len(neg))

# ------------------------------------------------------------------- passes
# detect_fn is swappable (default detect_pv) so CELL 5 can run the identical planting /
# matching logic against a different detector -- e.g. detect_perch_only -- and get a
# directly comparable number rather than a separately-implemented, possibly-diverging one.
def eval_pass(enc, ctx, seed, detect_fn=detect_pv):
    r = np.random.default_rng(seed); cache = []
    for bed in eval_beds:
        y = bed.copy(); L = len(y); spans, planted, snrs = [], [], []
        for k in range(CALLS_PER_BED):
            snr = SNR_LEVELS[k % len(SNR_LEVELS)]
            placed = None
            for _ in range(80):
                pos = int(r.integers(0, L-CLEN))
                if any(pos < e+int(0.4*SR) and pos+CLEN > s-int(0.4*SR) for s, e in spans): continue
                placed = pos; break
            if placed is None: continue
            c = ctx["target_pool"][int(r.integers(len(ctx["target_pool"])))][:CLEN]
            if len(c) < CLEN: c = np.pad(c, (0, CLEN-len(c)))
            g = rms(y[placed:placed+CLEN])*(10**(snr/20))/(rms(c)+1e-9)
            y[placed:placed+CLEN] += c.astype("float32")*g
            spans.append((placed, placed+CLEN))
            planted.append((placed/SR, (placed+CLEN)/SR)); snrs.append(snr)
        cache.append((detect_fn(enc, ctx, y, r), planted, snrs))
    return cache

def spec_pass(enc, ctx, seed, snr_db=6.0, detect_fn=detect_pv):
    """Specificity control. snr_db is now a parameter: v1 fixed it at +6 dB, so species
    discrimination had only ever been measured on loud, easy calls (roadmap 3.2)."""
    r = np.random.default_rng(seed); cache = []
    for bed in eval_beds:
        y = bed.copy(); L = len(y); spans, items = [], []
        plan = [("target", ctx["target_pool"])]*6
        for n_, p_ in ctx["decoy_pools"].items(): plan += [(n_, p_)]*3
        for kind, pool in plan:
            placed = None
            for _ in range(80):
                pos = int(r.integers(0, L-CLEN))
                if any(pos < e+int(0.4*SR) and pos+CLEN > s-int(0.4*SR) for s, e in spans): continue
                placed = pos; break
            if placed is None or not pool: continue
            c = pool[int(r.integers(len(pool)))][:CLEN]
            if len(c) < CLEN: c = np.pad(c, (0, CLEN-len(c)))
            g = rms(y[placed:placed+CLEN])*(10**(snr_db/20))/(rms(c)+1e-9)
            y[placed:placed+CLEN] += c.astype("float32")*g
            spans.append((placed, placed+CLEN))
            items.append((kind, (placed/SR, (placed+CLEN)/SR)))
        cache.append((detect_fn(enc, ctx, y, r), items))
    return cache

def neg_pass(enc, ctx, seed, detect_fn=detect_pv):
    """PURE NEGATIVE control -- unmodified beds, zero target calls planted.
    Every detection here is a false alarm by construction. This is the deployment-
    relevant number (false alarms per hour of clean audio) and v1 never measured it."""
    r = np.random.default_rng(seed + 500); cache = []
    for bed in eval_beds:
        cache.append(detect_fn(enc, ctx, bed.copy(), r))
    return cache

# ------------------------------------------------------------------- scoring
def score_eval(fuse, cache):
    scored, n_truth, by_snr = [], 0, []
    for dets, planted, snrs in cache:
        ev = [(s, e, float(fuse(p, v))) for s, e, p, v in dets]
        scored += match(ev, planted, IOU_THR); n_truth += len(planted)
        by_snr += list(zip(snrs, match_truth(ev, planted, IOU_THR)))
    m = summarise(scored, n_truth)
    m["fp_per_tp@R50"] = fp_per_tp(scored, n_truth, 0.5)
    m["recall_by_snr"] = {s: float(np.mean([sc >= m["thr"] for ss, sc in by_snr if ss == s]))
                          for s in SNR_LEVELS}
    return m

def score_spec(fuse, thr, cache, decoy_names):
    tg, dc = [], {k: [] for k in decoy_names}
    for dets, items in cache:
        ev = [(s, e, float(fuse(p, v))) for s, e, p, v in dets]
        for kind, span in items:
            best = max((iou(e[:2], span), e[2]) for e in ev) if ev else (0.0, 0.0)
            sc = best[1] if best[0] > IOU_THR else 0.0
            (tg if kind == "target" else dc[kind]).append(sc)
    t = float(np.mean([s >= thr for s in tg])) if tg else 0.0
    d = {k: float(np.mean([s >= thr for s in v])) for k, v in dc.items() if v}
    md = float(np.mean(list(d.values()))) if d else 0.0
    all_dc = [s for v in dc.values() for s in v]
    return t, md, (md/t if t else float('nan')), d, _auc(tg, all_dc)

def spec_coverage(cache):
    """-> fraction of planted items stage 1 LOCALISED at all (IoU > IOU_THR), by kind.

    Only the SNR sweep (roadmap 3.2) needs this, and there it is not optional. score_spec
    assigns 0.0 to any planted item that was never localised, and _auc counts a 0.0-vs-0.0
    pair as a tie worth 0.5. So an AUC drifting toward 0.50 at low SNR is ambiguous between
    two different findings: "target and decoy became indistinguishable" and "nothing was
    detected, so everything tied". Coverage separates them. Reporting the SNR curve without
    it would invite exactly the misreading the curve is meant to settle.

    Fusion-independent by construction -- a fusion rule changes an event's score, never its
    span -- so this is computed once per cache rather than once per fusion rule.
    """
    n = {"target": 0, "decoy": 0}; hit = {"target": 0, "decoy": 0}
    for dets, items in cache:
        for kind, span in items:
            k = "target" if kind == "target" else "decoy"
            n[k] += 1
            best = max((iou((s, e), span) for s, e, _p, _v in dets), default=0.0)
            hit[k] += int(best > IOU_THR)
    return {k: (hit[k]/n[k] if n[k] else float('nan')) for k in n}

def score_spec_detected(fuse, cache):
    """-> (AUC over LOCALISED items only, n_target_localised, n_decoy_localised).

    The companion to spec_coverage, and the direct measurement it was a proxy for.
    score_spec's AUC mixes two effects that pull in the same direction at low SNR: genuine
    species confusion, and stage 1 failing to localise the call at all -- which forces a
    0.0 and drags the AUC toward 0.50 for a reason that has nothing to do with species
    identity. Excluding unlocalised items (rather than scoring them 0.0) separates the two,
    so "specificity degraded" and "detection degraded" can be told apart from the numbers
    instead of argued about afterwards.

    Returns nan when either side is empty -- at low enough SNR that is the honest answer,
    and it must not be silently averaged in as if it were a measurement.
    """
    tg, dc = [], []
    for dets, items in cache:
        ev = [(s, e, float(fuse(p, v))) for s, e, p, v in dets]
        for kind, span in items:
            best = max((iou(e[:2], span), e[2]) for e in ev) if ev else (0.0, 0.0)
            if best[0] <= IOU_THR: continue      # never localised -> excluded, NOT zeroed
            (tg if kind == "target" else dc).append(best[1])
    return _auc(tg, dc), len(tg), len(dc)

def score_neg(fuse, thr, cache):
    """-> false alarms per hour on audio containing no target calls."""
    fa = sum(sum(1 for s, e, p, v in dets if float(fuse(p, v)) >= thr) for dets in cache)
    hours = (len(cache) * BED_SEC) / 3600.0
    return fa / hours if hours else float('nan')

# ------------------------------- stage-1 encoder (ALL targets excluded -- the bug fix)
def build_banks(exclude, features=FEATURES, seed=0, max_species=40):
    """One bank per front-end, sharing a single pass over the audio.

    An encoder trained on PCEN cannot be evaluated on log-Mel inputs, so the front-end
    cannot be a same-encoder ladder rung -- it needs its own encoder. But the expensive
    part is decoding audio, not featurising it, so each recording is loaded ONCE and
    featurised for every front-end. Cost is one extra encoder training, not a second
    pass over the dataset.
    """
    rng = np.random.default_rng(seed)
    sps = [s for s in meta.primary_label.unique() if s not in exclude]
    rng.shuffle(sps)
    banks = {f: {} for f in features}
    for sp in sps:
        if len(banks[features[0]]) >= max_species: break
        segs = {f: [] for f in features}
        for fn in meta[meta.primary_label == sp]["filename"].tolist()[:12]:
            try:
                y = ld(os.path.join(AUD, fn), 15)
                c = loud_off(y)//HOP + SEG_FR//2
                for f in features:
                    segs[f].append(crop(feat(y, f), c))
            except Exception:
                pass
        if len(segs[features[0]]) >= 8:
            for f in features:
                banks[f][sp] = segs[f]
    return banks

print(f"\nbuilding encoder banks for {list(FEATURES)}, excluding all {len(TARGET_SET)} "
      f"targets: {sorted(TARGET_SET)}")
banks = build_banks(TARGET_SET, FEATURES, seed=0)
for _f, _b in banks.items():
    assert not (TARGET_SET & set(_b)), \
        f"LEAKAGE: target species present in {_f} encoder bank: {TARGET_SET & set(_b)}"
    print(f"  {_f:<7} bank: {len(_b)} species, none of them targets (assertion passed)")

# Keyed by (feature, seed): a rung that switches the front-end must switch the encoder
# with it, or it is measuring an encoder/feature mismatch rather than the front-end.
encoders = {}
for _f in FEATURES:
    for es in ENCODER_SEEDS:
        print(f"training stage-1 encoder (feature={_f}, seed={es})...")
        torch.manual_seed(es)
        encoders[(_f, es)] = train_enc(banks[_f], np.random.default_rng(es))

if SKIP_MAIN_LOOP:
    print("SKIP_MAIN_LOOP=True -- stopping after CELL 2's encoder training.")
    print("res/wide/the per-species report do not exist this run. Proceed to CELL 4;")
    print("its own checkpointing resumes the ladder at the first unfinished config.")
else:
    # ------------------------------------------------------------------- main loop
    rows, t_start = [], time.time()
    for sp in TARGET_LIST:
        print(f"\n{'='*78}\nTARGET: {sp}\n{'='*78}")
        t_sp = time.time()
        try:
            ctx = build_target(sp)
        except Exception as ex:
            print(f"  SKIP {sp}: {type(ex).__name__}: {ex}"); continue

        for es in ENCODER_SEEDS:
            enc = encoders[(FEATURE, es)]
            per_seed = {name: [] for name in FUSIONS}
            for sd in SEEDS:
                t0 = time.time()
                ec = eval_pass(enc, ctx, sd)
                sc = spec_pass(enc, ctx, sd)
                nc = neg_pass(enc, ctx, sd)
                for name, fuse in FUSIONS.items():
                    m = score_eval(fuse, ec)
                    t, md, ratio, per, auc = score_spec(fuse, m["thr"], sc, ctx["DECOYS"])
                    per_seed[name].append(dict(
                        ap=m["ap"], f1=m["f1"], p=m["p"], r=m["r"],
                        fptp=m["fp_per_tp@R50"], auc=auc, ratio=ratio,
                        fa_per_hr=score_neg(fuse, m["thr"], nc),
                        fa_per_hr_r90=score_neg(fuse, m["thr_r90"], nc)))
                print(f"  seed {sd} done in {time.time()-t0:.0f}s")

            for name in FUSIONS:
                agg = {k: float(np.mean([d[k] for d in per_seed[name]])) for k in per_seed[name][0]}
                sds = {k + "_sd": float(np.std([d[k] for d in per_seed[name]], ddof=1))
                       if len(SEEDS) > 1 else 0.0 for k in per_seed[name][0]}
                rows.append({**ctx["covariates"], "encoder_seed": es, "fusion": name,
                             "feature": FEATURE, "n_seeds": len(SEEDS),
                             "use_timefilter": USE_TIMEFILTER, "use_hard_neg": USE_HARD_NEG,
                             **agg, **sds})
        print(f"  [{sp}] total {time.time()-t_sp:.0f}s")

    res = pd.DataFrame(rows)
    res.to_csv("/kaggle/working/argus_multispecies_results.csv", index=False)
    print(f"\nwrote {len(res)} rows -> argus_multispecies_results.csv "
          f"| wall clock {(time.time()-t_start)/60:.1f} min")

    # ------------------------------------------------------------------- headline table
    BASE, CAND = "stage-1 only  (p)", "perch only    (v)"
    print(f"\n{'='*100}\nSPECIFICITY AUC BY SPECIES  (threshold-free; 0.50 = target "
          f"indistinguishable from decoys)\n{'='*100}")
    print(f"{'species':<12}{'n_rec':>6}{'decoy_sim':>10}{'stereo':>8}"
          f"{'AUC stage-1':>13}{'AUC verified':>14}{'delta':>8}{'F1 s1':>7}{'F1 ver':>8}")
    print("-"*100)
    # Average over encoder seeds first, so a multi-encoder-seed run reports the mean rather
    # than silently taking whichever row happened to land first.
    NUM = ["auc", "f1", "fa_per_hr", "fa_per_hr_r90", "n_recordings",
           "decoy_sim_top1", "stereotypy"]
    piv = (res[res.fusion.isin([BASE, CAND])]
           .groupby(["species", "fusion"], sort=False)[NUM].mean().reset_index())
    wide = piv.pivot(index="species", columns="fusion")

    for sp in [s for s in TARGET_LIST if s in wide.index]:
        r = wide.loc[sp]
        print(f"{sp:<12}{int(r[('n_recordings', BASE)]):>6}{r[('decoy_sim_top1', BASE)]:>10.3f}"
              f"{r[('stereotypy', BASE)]:>8.3f}{r[('auc', BASE)]:>13.3f}{r[('auc', CAND)]:>14.3f}"
              f"{r[('auc', CAND)] - r[('auc', BASE)]:>+8.3f}"
              f"{r[('f1', BASE)]:>7.3f}{r[('f1', CAND)]:>8.3f}")

    # ---------------------------------------------------------- encoder-seed spread
    # The table above averages over encoder seeds -- which hides exactly the variance
    # ENCODER_SEEDS was widened to measure. Same bank, same data, same eval seeds; the only
    # thing differing is the encoder training run. Anything smaller than this spread is not
    # a real effect, no matter what a within-run SE says about it.
    if len(ENCODER_SEEDS) > 1:
        print(f"\n{'='*100}\nENCODER-SEED SPREAD -- pure training-run variance "
              f"(same bank, same data, {len(ENCODER_SEEDS)} encoder seeds)\n{'='*100}")
        es_piv = (res[res.fusion.isin([BASE, CAND])]
                  .groupby(["species", "fusion", "encoder_seed"])["auc"].mean().reset_index())
        worst = 0.0
        for sp in [s for s in TARGET_LIST if s in set(es_piv.species)]:
            for fus, tag in ((BASE, "stage-1"), (CAND, "verified")):
                g = es_piv[(es_piv.species == sp) & (es_piv.fusion == fus)].sort_values("encoder_seed")
                if g.empty: continue
                v = g.auc.to_numpy(); spread = float(v.max() - v.min())
                worst = max(worst, spread)
                print(f"  {sp:<10} {tag:<9} " + "  ".join(f"{x:.3f}" for x in v)
                      + f"   | spread {spread:+.3f}  sd {v.std(ddof=1):.3f}")
        print(f"\n  Largest spread from encoder training alone: {worst:.3f}")
        print(f"  Treat any AUC difference below ~{worst:.3f} as noise, not signal.")

    if len(wide) > 1:
        d = (wide[("auc", CAND)] - wide[("auc", BASE)]).to_numpy()
        se = d.std(ddof=1)/np.sqrt(len(d))
        print(f"\nACROSS SPECIES (n={len(d)}): mean AUC gain {d.mean():+.3f}, SE {se:.3f}, "
              f"delta/SE {d.mean()/se if se else float('nan'):.2f}   (|d/SE| < 2 is NOT evidence)")
        print(f"  verification helped:            {int((d > 0).sum())}/{len(d)} species")
        print(f"  below chance WITHOUT verifying: "
              f"{int((wide[('auc', BASE)] < 0.5).sum())}/{len(d)} species")
        print(f"  below chance WITH verifying:    "
              f"{int((wide[('auc', CAND)] < 0.5).sum())}/{len(d)} species")
        # The pre-registered outcome test (roadmap sec.2): does the original single-species
        # finding generalise, hold only conditionally, or fail? Reported either way.
        frac = (d > 0).mean()
        print("  -> outcome " + ("A (generalises)" if frac >= 0.8 else
                                 "B (conditional -- find what separates them)" if frac >= 0.4 else
                                 "C/D (does NOT generalise -- rebuild narrative around the boundary)"))

    print(f"\nFALSE ALARMS PER HOUR on clean audio (no target present)")
    print(f"{'species':<12}{'best-F1 thr':>24}{'recall-first thr':>26}")
    for sp in [s for s in TARGET_LIST if s in wide.index]:
        r = wide.loc[sp]
        print(f"  {sp:<10}{r[('fa_per_hr', BASE)]:>10.1f} ->{r[('fa_per_hr', CAND)]:>8.1f}"
              f"{r[('fa_per_hr_r90', BASE)]:>16.1f} ->{r[('fa_per_hr_r90', CAND)]:>8.1f}")


In [ ]:
# ===== CELL D / DEMO SNAPSHOT (current pipeline) -- run AFTER Cell 1 and Cell 2 =====
# Writes /kaggle/working/argus_demo_snapshot_v2.json, which site/index.html replays offline
# via "Load snapshot". No web server and no tunnel, so nothing here depends on the network.
#
# Same design as the July demo: real held-out White-bellied Sholakili calls (never used as
# the five examples) planted at known times into real Western Ghats soundscapes, one call per
# loudness level from +12 dB down to -3 dB, so the hard cases stay visible.
#
# One change, for honesty: the detection thresholds are chosen on CALIBRATION soundscapes and
# then applied unchanged to different soundscapes that are shown. That is the paper's protocol.
# The July demo picked its threshold on the very soundscapes it displayed, which flatters it.
import io, json, time, base64
import numpy as np, librosa, soundfile as sf
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

DEMO_SPECIES = "whbsho3"
DEMO_SEC     = 120.0
DEMO_SNRS    = (12, 9, 6, 3, 0, -3)
N_SHOW, N_CALIB = 3, 3
OUT_JSON     = "/kaggle/working/argus_demo_snapshot_v2.json"

t_demo = time.time()
assert DEMO_SPECIES in TARGET_LIST, f"{DEMO_SPECIES} is not a pre-registered target"
demo_ctx = build_target(DEMO_SPECIES)
demo_enc = encoders[(FEATURE, ENCODER_SEEDS[0])]
print(f"[demo] {DEMO_SPECIES}: {len(demo_ctx['support'])} support views, "
      f"{len(demo_ctx['target_pool'])} held-out calls, decoys {demo_ctx['DECOYS']}")


def wav_b64(y, sr=SR):
    buf = io.BytesIO()
    sf.write(buf, np.clip(np.asarray(y, dtype="float32"), -1, 1), sr, format="WAV", subtype="PCM_16")
    return base64.b64encode(buf.getvalue()).decode()


def spec_b64(y, sr=SR):
    # No axes or padding: the image spans exactly 0 -> duration, so the page can place
    # markers by percentage and have them line up.
    S = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=1024, hop_length=256)), ref=np.max)
    keep = librosa.fft_frequencies(sr=sr, n_fft=1024) <= 11000
    fig = plt.figure(figsize=(16, 4.2), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1]); ax.axis("off")
    ax.imshow(S[keep], origin="lower", aspect="auto", cmap="magma", vmin=-70, vmax=0)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", transparent=True, pad_inches=0); plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()


def build_scenario(bed, seed):
    r = np.random.default_rng(seed)
    n = int(DEMO_SEC * SR)
    off = int(r.integers(0, max(1, len(bed) - n)))
    y = bed[off:off + n].copy()
    pool = demo_ctx["target_pool"]
    spans, truth = [], []
    for snr in DEMO_SNRS:
        placed = None
        for _ in range(120):
            pos = int(r.integers(0, len(y) - CLEN))
            if any(pos < e + int(1.2 * SR) and pos + CLEN > s - int(1.2 * SR) for s, e in spans):
                continue
            placed = pos; break
        if placed is None:
            continue
        c = pool[int(r.integers(len(pool)))][:CLEN]
        if len(c) < CLEN: c = np.pad(c, (0, CLEN - len(c)))
        g = rms(y[placed:placed + CLEN]) * (10 ** (snr / 20)) / (rms(c) + 1e-9)
        y[placed:placed + CLEN] += c.astype("float32") * g
        spans.append((placed, placed + CLEN))
        truth.append({"start": placed / SR, "end": (placed + CLEN) / SR, "snr": int(snr)})
    peak = float(np.max(np.abs(y))) or 1.0
    if peak > 0.99: y = y * (0.99 / peak)
    return y.astype("float32"), sorted(truth, key=lambda t: t["start"])


def make(beds, base_seed, n, label):
    out = []
    for i in range(n):
        y, truth = build_scenario(beds[i % len(beds)], seed=base_seed + i)
        out.append({"id": i, "name": f"{label} #{i + 1}", "seconds": round(len(y) / SR, 1),
                    "audio": y, "truth": truth})
    return out


CALIB = make(calib_beds, 200, N_CALIB, "calibration soundscape")
SHOW = make(eval_beds, 100, N_SHOW, "Western Ghats soundscape")

for s in CALIB + SHOW:
    t0 = time.time()
    dets = detect_pv(demo_enc, demo_ctx, s["audio"], np.random.default_rng(0))
    # Round once, here, and compute every score from the rounded values -- the page does the
    # same arithmetic in JavaScript, so a borderline call cannot flip between the two.
    s["raw"] = [(float(a), float(b), round(float(p), 4), round(float(v), 4)) for a, b, p, v in dets]
    print(f"[demo] {s['name']}: {len(dets)} candidates in {time.time() - t0:.0f}s")

FUSE = {"p": lambda p, v: p, "v": lambda p, v: v, "pv": lambda p, v: p * v}


def match(events, truth):
    pairs = sorted(((iou((e[0], e[1]), (t["start"], t["end"])), i, j)
                    for i, e in enumerate(events) for j, t in enumerate(truth)), reverse=True)
    ue, ut = set(), set()
    for ov, i, j in pairs:
        if ov <= IOU_THR: break
        if i in ue or j in ut: continue
        ue.add(i); ut.add(j)
    return ue, ut


def tally(scenes, fn, thr):
    tp = fp = nt = 0
    for s in scenes:
        keep = [(a, b, fn(p, v)) for a, b, p, v in s["raw"] if fn(p, v) >= thr]
        ue, ut = match(keep, s["truth"])
        tp += len(ut); fp += len(keep) - len(ue); nt += len(s["truth"])
    return tp, fp, nt


def best_threshold(fn):
    scores = sorted({fn(p, v) for s in CALIB for _, _, p, v in s["raw"]})
    best, best_f1 = 1.0, -1.0
    for thr in scores:
        tp, fp, nt = tally(CALIB, fn, thr)
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / nt if nt else 0.0
        f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        if f1 > best_f1: best_f1, best = f1, thr
    return best, best_f1


THR = {}
for k, fn in FUSE.items():
    THR[k], f1 = best_threshold(fn)
    print(f"[demo] threshold[{k}] = {THR[k]:.4f}  (best F1 {f1:.3f}, chosen on CALIBRATION soundscapes only)")

print("[demo] results on the SHOWN soundscapes at the calibrated p x v threshold:")
for s in SHOW:
    tp, fp, nt = tally([s], FUSE["pv"], THR["pv"])
    print(f"   {s['name']}: {tp}/{nt} found, {fp} false alarms")
tp, fp, nt = tally(SHOW, FUSE["pv"], THR["pv"])
print(f"   ALL SHOWN: {tp}/{nt} found, {fp} false alarms")

sup = demo_ctx["support"]
one_per_shot = sup[1::3] if len(sup) >= 15 else sup[:5]   # the unshifted view of each recording
snapshot = {
    "target": DEMO_SPECIES,
    "support": {"count": len(one_per_shot), "clips": [wav_b64(w) for w in one_per_shot[:5]],
                "note": "The only labelled examples the model is given. Everything else is found unaided."},
    "analysis": None,
    "note": ("Current ARGUS pipeline (two-stage, Perch v2 verifier). Thresholds chosen on separate "
             "calibration soundscapes, then applied unchanged to the soundscapes shown."),
    "scenarios": [],
}
for s in SHOW:
    cands = []
    for a, b, p, v in s["raw"]:
        lo, hi = int(a * SR), int(b * SR)
        clip = s["audio"][max(0, lo):min(len(s["audio"]), hi)]
        if len(clip) < int(0.4 * SR): clip = np.pad(clip, (0, int(0.4 * SR) - len(clip)))
        cands.append({"start": round(a, 2), "end": round(b, 2), "p": p, "v": v, "audio": wav_b64(clip)})
    cands.sort(key=lambda c: c["start"])
    snapshot["scenarios"].append({"id": s["id"], "name": s["name"], "seconds": s["seconds"],
                                  "audio": wav_b64(s["audio"]), "spectrogram": spec_b64(s["audio"]),
                                  "candidates": cands, "truth": s["truth"], "thresholds": THR})
with open(OUT_JSON, "w") as fh:
    json.dump(snapshot, fh)
import os
print(f"[demo] wrote {OUT_JSON} ({os.path.getsize(OUT_JSON) / 1e6:.1f} MB) in {(time.time() - t_demo) / 60:.1f} min")
